In [6]:
import string
import random
import os
os.makedirs("test_output", exist_ok = True)
alphabet = string.ascii_lowercase

In [7]:
# Function to shift a character by a given amount
def shift_char(ch,shift):
    if ch.islower():
        base = ord("a")     # ord("a") finds the ASCII number for "a" 
    else:
        base = ord("A")     # If the letter is uppercase we take the ASCII for "A"
    return chr((ord(ch) - base + shift) % 26 + base)  # Returns the shifted letter. % 26 because we need the alphabet to wrap around

# Caesar cypher encryption and decryption
def caesar_encrypt(message, shift):
    if not message.isalpha():
        raise ValueError("The message must contain only letters.")
    if not 0 <= shift <= 25:        # A shift of 26 is just a shift of 0
        raise ValueError("Error, shift must be between 0 and 25.")
    return "".join(shift_char(ch, shift) for ch in message)     # Shifts each character in message and joins them with no spaces

def caesar_decrypt(ciphertext, shift):
    if not ciphertext.isalpha():
        raise ValueError("The ciphertext must contain only letters")
    if not 0 <= shift <= 25:
        raise ValueError("Error, shift must be between 0 and 25.")
    return "".join(shift_char(ch, -shift) for ch in ciphertext) # -shift since we are going back to what it was, so subtract the shift.

print(caesar_encrypt("hellohowareyou", 20))
print(caesar_decrypt(caesar_encrypt("hellohowareyou", 20), 20))

byffibiqulysio
hellohowareyou


In [8]:
# Function to convert the characters in the key to individual shifts
def key_to_shifts(key):
    keylist = []
    for ch in key:
        keylist.append(ord(ch.lower()) - ord("a"))  # ch.lower makes ch lowercase so that "KEY" and "key" have the same output
    return keylist

def vigenere_encrypt(message, key):
    if not message.isalpha():
        raise ValueError("The message must contain only letters.")
    if not key or not key.isalpha():    # The key needs to be non-empty and be only alphabetical characters
        raise ValueError("There was an error with the key.")
    shifts = key_to_shifts(key)
    return "".join(shift_char(message[i], shifts[i % len(key)]) for i in range(len(message)))     # Shifts each character from message with the correct shift acquired from the key and joins them

def vigenere_decrypt(ciphertext, key):
    if not ciphertext.isalpha():
        raise ValueError("The ciphertext must contain only letters")
    if not key or not key.isalpha():
        raise ValueError("There was an error with the key.")
    shifts = key_to_shifts(key)
    return "".join(shift_char(ciphertext[i], -shifts[i % len(key)]) for i in range(len(ciphertext)))      # -shifts because we need to shift backwards to the original text

print(vigenere_encrypt("hello", "key"))
print(vigenere_decrypt(vigenere_encrypt("hello", "key"), "key"))

rijvs
hello


In [9]:
# Brief asks that encryption and decryption be implemented to and from files

# Strips everything but letters in a text
def extract_text(text):
    return "".join(ch for ch in text if ch in string.ascii_letters)

# Opens a file and returns the text in it as a string
def read_file(path):
    with open(path, "r", encoding = "utf-8") as f:
        return f.read()

# Saves a string to a file
def write_file(path, text):
    with open(path, "w", encoding = "utf-8") as f:
        f.write(text)

# Define functions which perform the two forms of encryption and decryption on text files
def caesar_encrypt_file(input_file, output_file, shift):
    file_to_string =  read_file(input_file)
    write_file(output_file, caesar_encrypt(extract_text(file_to_string), shift))

def caesar_decrypt_file(input_file, output_file, shift):
    file_to_string =  read_file(input_file)
    write_file(output_file, caesar_decrypt(file_to_string, shift))

def vigenere_encrypt_file(input_file, output_file, key):
    file_to_string =  read_file(input_file)
    write_file(output_file, vigenere_encrypt(extract_text(file_to_string), key))

def vigenere_decrypt_file(input_file, output_file, key):
    file_to_string =  read_file(input_file)
    write_file(output_file, vigenere_decrypt(file_to_string, key))

In [10]:
# Random shifts for Caesar cypher testing
def random_caesar_shift():
    return random.randint(0,25)

# Random keys for Vigenere cypher testing
def random_vigenere_keys(length):
    return "".join(random.choice(alphabet) for _ in range(length))

# Returns a random message made up of lower and uppercase letters of a specified length
def random_message(length):
    return "".join(random.choice(string.ascii_letters) for _ in range(length))

def test_random(trials = 100):
    for _ in range(trials):
        r_msg = random_message(random.randint(1, 200))
        r_shift = random_caesar_shift()
        r_key = random_vigenere_keys(random.randint(1, len(r_msg)))
        assert caesar_decrypt(caesar_encrypt(r_msg, r_shift), r_shift) == r_msg
        assert vigenere_decrypt(vigenere_encrypt(r_msg, r_key), r_key) == r_msg
        assert vigenere_encrypt(r_msg, alphabet[r_shift]) == caesar_encrypt(r_msg, r_shift)   # A single-letter key should behave like the Caesar cypher
    print(f"Random tests: passed {trials} trials.")


def test_caesar(message):
    for shift in range(26):
        assert caesar_decrypt(caesar_encrypt(message, shift), shift) == message
    print(f"Caesar: Passed all 26 shifts.")

test_caesar("HelloWorld")
test_random()

def test_files(file):
    original = extract_text(read_file(file))
    r_shift = random_caesar_shift()
    caesar_encrypt_file(file, "test_output/caesar_encrypted.txt", r_shift)
    caesar_decrypt_file("test_output/caesar_encrypted.txt", "test_output/caesar_decrypted.txt", r_shift)
    assert read_file("test_output/caesar_decrypted.txt") == original

    r_key = random_vigenere_keys(random.randint(1, len(original)))
    vigenere_encrypt_file(file, "test_output/vigenere_encrypted.txt", r_key)
    vigenere_decrypt_file("test_output/vigenere_encrypted.txt", "test_output/vigenere_decrypted.txt", r_key)
    assert read_file("test_output/vigenere_decrypted.txt") == original
    print(f"File tests passed for {file}")

test_files("message_moby_dick.txt")
test_files("message_wuthering_heights.txt")

Caesar: Passed all 26 shifts.
Random tests: passed 100 trials.
File tests passed for message_moby_dick.txt
File tests passed for message_wuthering_heights.txt


In [52]:
#create empty dictionary to record letter occurrences and frequencies
freqs_all = {}

def freq_calc(input_file, decrypt_file=True):
    occurrences = {}
    freqs_single = {}
    #extract text string from file
    file_string = extract_text(read_file(input_file))
    file_string = file_string.lower()

    #record letter frequencies in dictionary
    for i in range(len(file_string)):
        if file_string[i] not in occurrences:
            occurrences[file_string[i]] = 1
        else:
            occurrences[file_string[i]] += 1
    
    #calculate frequencies for that file
    for vals in occurrences:
        freqs_single[vals] = occurrences[vals] / len(file_string)

    if decrypt_file:
        #take the mean with other files
        for vals in freqs_single:
            if vals not in freqs_all:
                freqs_all[vals] = freqs_single[vals]
            else:
                freqs_all[vals] = (freqs_all[vals] + freqs_single[vals])/2
        return freqs_all
    else:
        return freqs_single

In [53]:
freq_calc("message_emma.txt")
freq_calc("message_heart_of_darkness.txt")
freq_calc("message_moby_dick.txt")
freq_calc("message_tomorrows_tangle.txt")
freq_calc("message_treasure_island.txt")
freq_calc("message_wuthering_heights.txt")

{'e': 0.12625681634725153,
 'm': 0.026212570243520087,
 'a': 0.08193672451484009,
 'w': 0.022429562334091775,
 'o': 0.07414169628010706,
 'd': 0.04570674130313672,
 'h': 0.06272998407423436,
 'u': 0.02875899853733271,
 's': 0.06330597926551593,
 'n': 0.07159044588805132,
 'c': 0.024054529083581327,
 'l': 0.041079836491678685,
 'v': 0.010421152099841478,
 'r': 0.055999138460213885,
 'i': 0.06751844371867835,
 't': 0.08801259793630814,
 'f': 0.02354817093061345,
 'b': 0.014739388726109168,
 'p': 0.018894367386455262,
 'y': 0.01903518270372242,
 'g': 0.02162077812534602,
 'x': 0.0014911174997897642,
 'q': 0.0009100691926628047,
 'j': 0.0011402305871689373,
 'k': 0.00811198201627624,
 'z': 0.0003534962534725053}